In [ ]:
# Choose the model, pipeline type and current execution to debug.
# The notebook reads that job's actual inputs, including cached producer paths.
# The historical examples below belong to namespace src1007cced-i1102a.
MODEL_ID = "usecase2"
PIPELINE_KIND = "testing"  # "testing" or "inference"
EXECUTION_ID = "qaq3ddxjfasy"
# Historical inference example: PIPELINE_KIND = "inference"; EXECUTION_ID = "brf4bz1lunno"
# Use an execution from the namespace in your notebooks/config.json.


In [ ]:
# Locate this checkout and inspect the existing Postprocess job; no pipeline is run.
# Show each input's physical producer separately from the requested execution.
# Keep the printed assignments usable on their own; the final cell exports a full VS Code block.
from pathlib import Path
import boto3
from botocore.exceptions import ClientError
import json
import sys

# Discard this notebook's previous script before resolving another run.
previous_script_directory = globals().pop("postprocess_script_directory", None)
if previous_script_directory is not None:
    previous_script_directory.cleanup()
# Do not reuse a previous run's inputs or export after a failed lookup.
for name in ("DATA_URI", "SELECTION_URI", "MODEL_URI", "SCORE_OUTPUT_URI", "REGION",
             "postprocess_script", "vscode_code"):
    globals().pop(name, None)
REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
source_directory = str(REPOSITORY / "mlops-models/src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)
config = json.loads((REPOSITORY / "notebooks/config.json").read_text())
if PIPELINE_KIND not in {"testing", "inference"}:
    raise ValueError("Choose testing or inference.")
sm = boto3.client("sagemaker")
def read_steps(arn):
    return [step for page in sm.get_paginator("list_pipeline_execution_steps").paginate(
        PipelineExecutionArn=arn) for step in page["PipelineExecutionSteps"]]


def missing_resource(error):
    details = error.response["Error"]
    return details["Code"] in {"ResourceNotFound", "ResourceNotFoundException"} or (
        details["Code"] == "ValidationException" and any(
            wording in details.get("Message", "").lower()
            for wording in ("does not exist", "could not find", "not found")
        )
    )


def find_execution(kind):
    pipeline = sm.describe_pipeline(PipelineName=f"{config['namespace']}-{kind}")
    arn = pipeline["PipelineArn"] + "/execution/" + EXECUTION_ID
    return arn, read_steps(arn)


try:
    execution_arn, steps = find_execution(PIPELINE_KIND)
except ClientError as original_error:
    if not missing_resource(original_error):
        raise
    other_kind = "inference" if PIPELINE_KIND == "testing" else "testing"
    try:
        other_arn, _ = find_execution(other_kind)
    except ClientError as other_error:
        if not missing_resource(other_error):
            raise RuntimeError(
                f"Selected execution was not found; could not check {other_kind!r}: "
                f"{other_error.response['Error']['Code']}. {other_error.response['Error'].get('Message', '')}"
            ) from other_error
        raise ValueError(
            f"Execution {EXECUTION_ID!r} was not found in the testing or inference "
            f"pipeline for namespace {config['namespace']!r}, region {sm.meta.region_name!r}. "
            "Check the execution ID, namespace and region."
        ) from original_error
    raise ValueError(
        f"Execution {EXECUTION_ID!r} was found in {other_kind!r}: {other_arn}. "
        f"Set PIPELINE_KIND = {other_kind!r} in the first cell, then rerun the input cell."
    ) from original_error

# Read Postprocess's job rather than guessing where cached producer outputs live.
lookup_arn = execution_arn
visited = set()
requested_steps = None
while True:
    if lookup_arn in visited:
        raise ValueError("Repeated cache reference: " + lookup_arn)
    visited.add(lookup_arn)
    if requested_steps is None:
        requested_steps = {step["StepName"]: step for step in steps}
    step = next((step for step in steps if step["StepName"] == "Postprocess"), None)
    if step is None:
        raise ValueError("No Postprocess step found for " + lookup_arn)
    job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
    if job_arn:
        break
    cached_from = step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn")
    if not cached_from:
        raise ValueError("Postprocess has no Processing job yet. Run it once before inspecting its inputs.")
    lookup_arn = cached_from
    steps = read_steps(lookup_arn)

job = sm.describe_processing_job(ProcessingJobName=job_arn.rsplit("/", 1)[1])
environment = job.get("Environment", {})
if environment.get("MODEL_ID") != MODEL_ID:
    raise ValueError(f"This job belongs to {environment.get('MODEL_ID')!r}, not {MODEL_ID!r}.")
if environment.get("MLOPS_PREDICTION_FORMAT", "csv") != "csv":
    raise ValueError("This notebook uses Processing/CSV scoring, not native Batch Transform.")
mounts = {item["S3Input"]["LocalPath"].rstrip("/"): item["S3Input"]["S3Uri"].rstrip("/")
          for item in job["ProcessingInputs"] if "S3Input" in item}
DATA_URI = mounts["/opt/ml/processing/raw"] + "/data.csv"
SELECTION_URI = mounts["/opt/ml/processing/selection"] + "/selected_features.csv"
MODEL_URI = mounts["/opt/ml/processing/model"]
SCORE_OUTPUT_URI = mounts["/opt/ml/processing/input"] + "/"
REGION = sm.meta.region_name


# Everything below is a Python comment or assignment, so it is also copyable.
print(f"# Current {PIPELINE_KIND.upper()} execution: {EXECUTION_ID} | model: {MODEL_ID}")
print(f"# Postprocess job: {job_arn.rsplit('/', 1)[1]}")
postprocess_cache = requested_steps["Postprocess"].get("CacheHitResult", {}).get("SourcePipelineExecutionArn")
if postprocess_cache:
    print(f"# Postprocess itself reused from: {postprocess_cache.rsplit('/', 1)[-1]}")
for label, step_name, uri in (
    ("Raw data", "GetData", DATA_URI),
    ("Selected features", "Preprocess", SELECTION_URI),
    ("Score files", "Score", SCORE_OUTPUT_URI),
    ("Model archive", None, MODEL_URI),
):
    _, has_run, remainder = uri.rpartition("/runs/")
    producer = remainder.split("/", 1)[0] if has_run else "not encoded in URI"
    cache = requested_steps.get(step_name, {}).get("CacheHitResult", {})
    cached_from = cache.get("SourcePipelineExecutionArn")
    status = f"; cache hit from {cached_from.rsplit('/', 1)[-1]}" if cached_from else ""
    print(f"# {label}: stored under execution {producer}{status}")
print("# Cached steps can read files stored under earlier execution IDs.")
print("# Raw data is Postprocess's input. Prepared data is Score's input.")
print("# The model's saved _execution.json identifies training origin after extraction.")
for name in ("MODEL_ID", "PIPELINE_KIND", "REGION", "DATA_URI", "SELECTION_URI",
             "MODEL_URI", "SCORE_OUTPUT_URI"):
    print(f"{name} = {globals()[name]!r}")


In [ ]:
# Keep the editable setup and execute call below in one plain-Python script.
# Write that exact script to a temporary file before running it with the local entrypoint.
# The final cell exports this file, even if execution fails; no notebook save is needed.
from pathlib import Path
from tempfile import TemporaryDirectory

# Each preparation owns one unique folder; rerunning removes only its predecessor.
previous_script_directory = globals().pop("postprocess_script_directory", None)
if previous_script_directory is not None:
    previous_script_directory.cleanup()
for name in ("postprocess_script", "vscode_code"):
    globals().pop(name, None)

# Edit the Python between the triple quotes. It is both the notebook run and the export.
postprocess_body = r"""# Download the raw rows, selection and model; copy every Score file with its relative path.
# Unpack the model and set the environment used by model code.
# Call the local checkout's entrypoint; keep results in a new local output directory.
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import json
import os
import tarfile
import boto3

work_directory = Path(mkdtemp(prefix="postprocess-"))
data_directory = work_directory / "raw"
selection_file = work_directory / "selection/selected_features.csv"
model_archive = work_directory / "model.tar.gz"
model_directory = work_directory / "model"
prediction_directory = work_directory / "input"
output_directory = work_directory / "output"
output_directory.mkdir()

# Local model reads and writable output; the entrypoint does not upload run reports.
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_MODEL_ARTIFACT_URI": MODEL_URI,
    "MLOPS_WRITE_RUN_REPORT": "0",
})

# =============================================================================
# DOWNLOAD INPUTS
# =============================================================================
# DATA_URI is raw, not Preprocess/prepared/data.csv.
s3 = boto3.client("s3", region_name=REGION)
for uri, destination in (
    (DATA_URI, data_directory / "data.csv"),
    (SELECTION_URI, selection_file),
    (MODEL_URI, model_archive),
):
    print(f"Download: {uri}", flush=True)
    destination.parent.mkdir(parents=True, exist_ok=True)
    address = urlsplit(uri)
    s3.download_file(address.netloc, address.path.lstrip("/"), str(destination))

# Copy the whole Score folder, including messaging, importance files and nested output/.
address = urlsplit(SCORE_OUTPUT_URI)
prefix = address.path.lstrip("/").rstrip("/") + "/"
score_files = 0
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=address.netloc, Prefix=prefix):
    for item in page.get("Contents", []):
        key = item["Key"]
        if key.endswith("/"):
            continue
        destination = (prediction_directory / key[len(prefix):]).resolve()
        destination.relative_to(prediction_directory.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(address.netloc, key, str(destination))
        score_files += 1
print(f"Score files: {score_files} | {prediction_directory}", flush=True)

# =============================================================================
# EXTRACT MODEL AND SHOW LOCAL PATHS
# =============================================================================
# The entrypoint receives an extracted directory, not model.tar.gz.
with tarfile.open(model_archive) as archive:
    archive.extractall(model_directory, filter="data")
training_record = model_directory / "_execution.json"
if training_record.is_file():
    training_origin = json.loads(training_record.read_text()).get("producer_execution_id") or "not recorded"
    print(f"Training producer recorded inside model: {training_origin}", flush=True)
else:
    print("Training producer not recorded inside this model archive.", flush=True)
print(json.dumps({
    "model_id": MODEL_ID,
    "data_directory": str(data_directory),
    "selection_file": str(selection_file),
    "input_path": str(prediction_directory),
    "model_dir": str(model_directory),
    "output_dir": str(output_directory),
}, indent=2), flush=True)

# =============================================================================
# EXECUTE POSTPROCESS
# =============================================================================
result_directory = execute(
    model_id=MODEL_ID,
    input_path=prediction_directory,
    output_dir=output_directory,
    model_dir=model_directory,
    data_directory=data_directory,
    selection_file=selection_file,
)
print(result_directory)
"""
entrypoint_name = "postprocess_test.py" if PIPELINE_KIND == "testing" else "postprocess_inference.py"
assignments = "\n".join(
    f"{name} = {globals()[name]!r}" for name in
    ("MODEL_ID", "PIPELINE_KIND", "REGION", "DATA_URI", "SELECTION_URI", "MODEL_URI", "SCORE_OUTPUT_URI")
)
postprocess_script_directory = TemporaryDirectory(prefix="postprocess-script-")
postprocess_script = Path(postprocess_script_directory.name) / entrypoint_name
postprocess_script.write_text(assignments + "\n\n" + postprocess_body, encoding="utf-8")
print(f"Run/export script: {postprocess_script}", flush=True)

# =============================================================================
# NOTEBOOK ENTRYPOINT IMPORT AND RUN
# =============================================================================
# This adapter is outside the saved script; the export calls execute in your entrypoint.
import importlib
if PIPELINE_KIND == "testing":
    from mlops_models.entrypoints import postprocess_test as postprocess
else:
    from mlops_models.entrypoints import postprocess_inference as postprocess
execute = importlib.reload(postprocess).execute
exec(compile(postprocess_script.read_text(encoding="utf-8"), str(postprocess_script), "exec"))


In [ ]:
# Export the exact temporary script prepared by the preceding cell, even if its run failed.
# Copy this whole output to the bottom of the named entrypoint, replacing its previous main block.
# Remove this notebook's temporary script after exporting; keep downloaded data/results for debugging.
from textwrap import indent

if "postprocess_script" not in globals():
    raise ValueError("Run the preceding execution cell first so it writes the temporary script.")
vscode_code = (
    f"# Paste at the bottom of {postprocess_script.name}; replace the previous local main block.\n"
    'if __name__ == "__main__":\n'
    + indent(postprocess_script.read_text(encoding="utf-8"), "    ")
)
compile(vscode_code, postprocess_script.name, "exec")
postprocess_script_directory.cleanup()
del postprocess_script_directory, postprocess_script
print(vscode_code)
